# 🧬 **LeetCode Bash Scripts for ROSALIND**

Welcome to the ultimate repository for solving **ROSALIND** bioinformatics challenges using the power of **Bash scripting**. 

Learning bioinformatics usually requires solving computational problems of varying difficulty that are extracted from real challenges of molecular biology. This repository is dedicated to tackling those problems head-on using lightweight, efficient, and native command-line tools.

---

🎯 **About ROSALIND**

> To make learning bioinformatics fun and easy, Rosalind was founded as a platform for learning bioinformatics through problem-solving. 
> 
> Rosalind offers an array of intellectually stimulating problems that grow in biological and computational complexity; each problem is checked automatically, so that the only resource required to learn bioinformatics is an internet connection. Rosalind also promises to facilitate improvements in standard bioinformatics education by providing a vital teaching aid and a central homework resource.

---

## Problem 21: Locating Restriction Sites
**Link** [https://rosalind.info/problems/revp/](https://rosalind.info/problems/revp/)

A DNA string is a **reverse palindrome** if it is equal to its reverse complement. For instance, `GCATGC` is a reverse palindrome because its reverse complement is `GCATGC`. See Figure.
![image](./DATA/REVP.png)

**Given:** A DNA string of length at most 1 kbp in FASTA format.

**Return:** The position and length of every reverse palindrome in the string having length between 4 and 12. You may return these pairs in any order.

In [6]:
%%bash
cat << EOF >data_bai21.txt
>Rosalind_24
TCAATGCATGCGGGTCTATATGCAT
EOF

**Cach 1**

In [7]:
%%bash

FILE="data_bai21.txt"
dna=""

# 1. Đọc file và nối chuỗi DNA (bỏ qua dòng tiêu đề)
while read -r line; do
    if [[ "${line:0:1}" != ">" ]]; then
        dna+="$line"
    fi
done < "$FILE"

len=${#dna}

# 2. Duyệt qua từng vị trí i (chỉ số 0-based trong Bash)
for (( i=0; i<len; i++ )); do
    # Chiều dài k chẵn từ 4 đến 12
    for (( k=4; k<=12; k+=2 )); do
        if (( i + k > len )); then
            break
        fi
        
        is_revp=1
        # 3. Kiểm tra tính chất reverse palindrome bằng cách so sánh 2 đầu ép vào giữa
        for (( j=0; j<k/2; j++ )); do
            c1="${dna:i+j:1}"
            c2="${dna:i+k-1-j:1}"
            
            # Kiểm tra bắt cặp Watson-Crick
            if [[ ! ( ( "$c1" == "A" && "$c2" == "T" ) || \
                      ( "$c1" == "T" && "$c2" == "A" ) || \
                      ( "$c1" == "C" && "$c2" == "G" ) || \
                      ( "$c1" == "G" && "$c2" == "C" ) ) ]]; then
                is_revp=0
                break
            fi
        done
        
        # Nếu đúng là reverse palindrome thì in ra (vị trí i+1 theo yêu cầu 1-based)
        if (( is_revp == 1 )); then
            echo "$((i + 1)) $k"
        fi
    done
done

4 6
5 4
6 6
7 4
17 4
18 4
20 6
21 4


**Cach 2**

In [8]:
%%bash
awk '
!/^>/ { dna = dna $0 }

END {
    len = length(dna)
    
    for (i = 1; i <= len; i++) {
        # Chiều dài k chẵn từ 4 đến 12
        for (k = 4; k <= 12; k += 2) {
            if (i + k - 1 > len) break
            
            is_revp = 1
            # Kiểm tra tính chất reverse palindrome từ 2 đầu của chuỗi con
            for (j = 0; j < k / 2; j++) {
                c1 = substr(dna, i + j, 1)
                c2 = substr(dna, i + k - 1 - j, 1)
                
                if (!((c1 == "A" && c2 == "T") || (c1 == "T" && c2 == "A") || \
                      (c1 == "C" && c2 == "G") || (c1 == "G" && c2 == "C"))) {
                    is_revp = 0
                    break
                }
            }
            
            if (is_revp) {
                print i, k
            }
        }
    }
}' data_bai21.txt

4 6
5 4
6 6
7 4
17 4
18 4
20 6
21 4


---
## Problem 22: RNA Splicing
**Link** [https://rosalind.info/problems/splc/](https://rosalind.info/problems/splc/)

After identifying the exons and introns of an RNA string, we only need to delete the introns and concatenate the exons to form a new string ready for translation.

**Given:** A DNA string $s$ (of length at most 1 kbp) and a collection of substrings of $s$ acting as introns. All strings are given in FASTA format.

**Return:** A protein string resulting from transcribing and translating the exons of $s$. (Note: Only one solution will exist for the dataset provided.)

In [ ]:
%%bash
cat << EOF >data_bai22.txt
>Rosalind_10
ATGGTCTACATAGCTGACAAACAGCACGTAGCAATCGGTCGAATCTCGAGAGGCATATGGTCACATGATCGGTCGAGCGTGTTTCAAAGTTTGCGCCTAG
>Rosalind_12
ATCGGTCGAA
>Rosalind_15
ATCGGTCGAGCGTGT
EOF

**Cách 1**

In [2]:
%%bash

# xử lý file đầu vào bằng cách cắt các đoạn intron trong trinh tự. data xử lý là bai22.txt

SEQ=$(awk 'NR==2 {print}' data_bai22.txt)
intron=($(awk '/>/ || NR==2 {next} {print $0}' data_bai22.txt))
echo -e "trình tự DNA: $SEQ\ntrình tự các đoạn intron: ${intron[@]}\n"

# sau khi đã lọc được data, chúng ta sẽ xóa các đoạn intron có trong SEQ
for intron in "${intron[@]}"; do
    SEQ=$(echo "$SEQ" | sed "s/$intron//g")
done
echo -e "trình tự sau khi xử lý: $SEQ\n"

# tiếp theo ta dịch mã đơn giản cho trình tự
translate "$(echo $SEQ | sed 's/T/U/g')"

trình tự DNA: ATGGTCTACATAGCTGACAAACAGCACGTAGCAATCGGTCGAATCTCGAGAGGCATATGGTCACATGATCGGTCGAGCGTGTTTCAAAGTTTGCGCCTAG
trình tự các đoạn intron: ATCGGTCGAA ATCGGTCGAGCGTGT

trình tự sau khi xử lý: ATGGTCTACATAGCTGACAAACAGCACGTAGCATCTCGAGAGGCATATGGTCACATGTTCAAAGTTTGCGCCTAG

MVYIADKQHVASREAYGHMFKVCA



**Cach 2:**

In [3]:
%%bash
# Bước 1: Dùng awk để đọc và gom các dòng DNA lại (bỏ qua dòng tiêu đề >)
sequences=($(awk '/^>/ {if (seq != "") print seq; seq=""; next} {seq=seq$0} END {print seq}' data_bai22.txt ))

# Chuỗi đầu tiên (vị trí 0) luôn là DNA chính
main_dna="${sequences[0]}"

# Bước 2: Vòng lặp tự động đọc từ vị trí số 1 đến hết (đây là các Intron)
for (( i=1; i<${#sequences[@]}; i++ )); do
    intron="${sequences[$i]}"

    # Tiến hành xóa đoạn intron này khỏi chuỗi DNA chính
    main_dna="${main_dna//$intron/}"
done
echo "$main_dna" > dna.txt

# Phiên mã
RNA="$(cat dna.txt)"
echo $RNA | tr 'T' 'U' > rna.txt
# Dịch mã
mA="$(cat rna.txt)"
translate ${mA:0}

MVYIADKQHVASREAYGHMFKVCA



---
## Problem 23: Enumerating k-mers Lexicographically
**Link:** [https://rosalind.info/problems/lgis/](https://rosalind.info/problems/lgis/)

Assume that an alphabet $\mathcal{A}$ has a predetermined order; that is, we write the alphabet as a permutation $\mathcal{A} = (a_1, a_2, \dots, a_k)$, where $a_1 < a_2 < \dots < a_k$. For instance, the English alphabet is organized as $(\text{A}, \text{B}, \dots, \text{Z})$.

Given two strings $s$ and $t$ having the same length $n$, we say that $s$ precedes $t$ in the **lexicographic order** (and write $s <_{\text{Lex}} t$) if the first symbol $s[j]$ that doesn't match $t[j]$ satisfies $s_j < t_j$ in $\mathcal{A}$.

**Given:** A collection of at most 10 symbols defining an ordered alphabet, and a positive integer $n$ ($n \le 10$).

**Return:** All strings of length $n$ that can be formed from the alphabet, ordered lexicographically (use the standard order of symbols in the English alphabet).

In [4]:
%%bash
cat << 'EOF' > data_bai23.txt
A C G T
2
EOF

# Đọc bảng chữ cái (dòng đâu tiên trong tệp) và đổi khoảng trắng thành dấu phẩy
a="{$(head -1 data_bai23.txt | tr ' ' ',')}"
# Đọc độ dài chuỗi (dòng thứ 2 trong tệp)
n=$(tail -1 data_bai23.txt)
# Lặp bảng chữ cái n lần, sinh mọi tổ hợp, rồi sắp xếp
eval echo "$(printf "$a%.0s" $(seq $n))" | tr ' ' '\n' | sort

AA
AC
AG
AT
CA
CC
CG
CT
GA
GC
GG
GT
TA
TC
TG
TT


---
## Problem 24: Longest Increasing Subsequence
**Link:** [https://rosalind.info/problems/iev/](https://rosalind.info/problems/lgis/)

A **subsequence** of a permutation is a collection of elements of the permutation in the order that they appear. For example, $(5, 3, 4)$ is a subsequence of $(5, 1, 3, 4, 2)$.

A subsequence is **increasing** if the elements of the subsequence increase, and **decreasing** if the elements decrease. For example, given the permutation $(8, 2, 1, 6, 5, 7, 4, 3, 9)$, an increasing subsequence is $(2, 6, 7, 9)$, and a decreasing subsequence is $(8, 6, 5, 4, 3)$. You may verify that these two subsequences are as long as possible.

**Given:** A positive integer $n \le 10000$ followed by a permutation $\pi$ of length $n$.

**Return:** A longest increasing subsequence of $\pi$, followed by a longest decreasing subsequence of $\pi$.


In [5]:
%%bash
{ read n; read line; } <<EOF
5
5 1 4 2 3
EOF
arr=($line)
for ((i=0; i<n; i++)); do
    lis[i]=${arr[i]}
    lds[i]=${arr[i]}
done
for ((i=0; i<n; i++)); do
for ((j=0; j<i; j++)); do
        len_lis_i=$(echo "${lis[i]}" | wc -w)
        len_lis_j=$(echo "${lis[j]}" | wc -w)
        len_lds_i=$(echo "${lds[i]}" | wc -w)
        len_lds_j=$(echo "${lds[j]}" | wc -w)
        if (( arr[j] < arr[i] )) && (( len_lis_j + 1 > len_lis_i )); then
            lis[i]="${lis[j]} ${arr[i]}"
            fi
        if (( arr[j] > arr[i] )) && (( len_lds_j + 1 > len_lds_i )); then
            lds[i]="${lds[j]} ${arr[i]}"
            fi
done
done
max_lis=""
max_lds=""

for ((i=0; i<n; i++)); do
    [[ $(echo "${lis[i]}" | wc -w) -gt $(echo "$max_lis" | wc -w) ]] && max_lis=${lis[i]}
    [[ $(echo "${lds[i]}" | wc -w) -gt $(echo "$max_lds" | wc -w) ]] && max_lds=${lds[i]}
done
echo "$max_lis"
echo "$max_lds"

1 2 3
5 4 2
